# Práctica: Solicitudes de Lectura JSON-RPC con Alchemy

En esta actividad aprenderás a realizar solicitudes de lectura a la blockchain de Ethereum utilizando el API de Alchemy y el protocolo JSON-RPC.

## Recursos recomendados

- Tutorial Hello World: [alchemyplatform/hello-world-tutorial](https://github.com/alchemyplatform/hello-world-tutorial/tree/main)
- Crea tu propia aplicación en [Alchemy Dashboard](https://dashboard.alchemy.com/) y sigue la guía paso a paso.
- Práctica guiada: [Alchemy University](https://university.alchemy.com/course/ethereum/md/614b9f3c7e426a001019be54)

## 📋 Requisitos

- Python 3.7+
- Node.js instalado (versión 14+)
- Como parte de la demo, opcional, una API key de Alchemy ([obtén una gratis aquí](https://www.alchemy.com/))

## 🔧 Verificación de Node.js

Primero, verifica que Node.js esté instalado:

## 🛠️ Resolución de problemas

Si encuentras problemas de entorno, prueba reiniciar el kernel de Jupyter (`Restart Kernel`). Verifica que las dependencias estén instaladas y que tu API key esté configurada correctamente en `.env`.

## ▶️ Ejecutar paso a paso

Los pasos son secuenciales; es necesario inicializar en muchos casos, por lo tanto, ejecuta celda a celda en orden para evitar errores, o al menos hazlo una vez. Por eso, no podrás ir directamente a una celda sin ejecutar una previa.

In [2]:
# Configurar entorno para Node.js

from src.setupNode import setup_node_path

# Esto detecta Node.js en nvm, conda, sistema, etc.
setup_node_path()

# Verificar Node.js
from src.setupNode import verify_environment

status = verify_environment()
for tool, available in status.items():
    icon = "✅" if available else "❌"
    state = 'Disponible' if available else 'No encontrado'
    print(f"{icon} {tool}: {state}")

🔍 Buscando Node.js...
✅ Node.js ya está disponible en PATH
✅ Node.js: v22.19.0
✅ npm: 10.9.3
✅ python: Disponible
✅ node: Disponible
✅ npm: Disponible


## 📦 Demo

Ahora ejecutemos un script que usa solo módulos built-in de Node.js para llamar al API de Alchemy.

**⚠️ ANTES DE EJECUTAR:** Actualiza `YOUR_API_KEY` en `.env` con tu API key de Alchemy.

In [8]:
from src.nodeRunner import NodeRunner
from src.nodeRunner import node_script
from src.nodeRunner import node_script_debug


# Crear runner con directorio actual
runner = NodeRunner('./alchemy-json-activity')

# Instalar paquetes necesarios
print('📦 Instalando dependencias necesarias...')
runner.install_packages('dotenv')
runner.install_packages('axios')

# Ejecutar script que usa solo módulos built-in
block_data = node_script('alchemy-json-activity/activity-JSON-RPC-Read-Requests.js')

✅ Node.js version: v22.19.0
📦 Instalando dependencias necesarias...
📦 Instalando paquetes npm: dotenv
✅ Paquetes instalados correctamente
📦 Instalando paquetes npm: axios
✅ Paquetes instalados correctamente
✅ Node.js version: v22.19.0
🚀 Ejecutando: alchemy-json-activity/activity-JSON-RPC-Read-Requests.js
[dotenv@17.2.3] injecting env (3) from .env -- tip: ⚙️  enable debug logging with { debug: true }
eth_getBlockByNumber 46147

output:

{
  "hash": "0x9f5303c4e06e573978156f63b334ef4923d8fe851e4574faa843c1097914eed6",
  "parentHash": "0xe6455a3355b273101dedb72f2b05e30806022e680e51716aa97481387383993c",
  "sha3Uncles": "0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a7413f0a142fd40d49347",
  "miner": "0x2f14582947e292a2ecd20c430b46f2d27cfe213c",
  "stateRoot": "0xdc7adac1b43457ce43dc57ff85f4c6ca6b3791fb463526acd29908eb48f5a7f2",
  "transactionsRoot": "0x56e81f171bcc55a6ff8345e692c0f86e5b48e01b996cadc001622fb5e363b421",
  "receiptsRoot": "0x56e81f171bcc55a6ff8345e692c0f86e5b48e01b996cadc001

Analizar resultado

In [7]:
import json
from src.utils import *
from datetime import datetime

# Parsear el JSON del resultado
if block_data:
    block = json.loads(block_data)
    
    print("=" * 80)
    print("📦 INFORMACIÓN DEL BLOQUE")
    print("=" * 80)
    
    # Definiciones de campos conocidos (descripción, convertir_hex)
    field_definitions = {
        'number': ('Número del bloque en la cadena', True),
        'hash': ('Hash único que identifica este bloque', False),
        'parentHash': ('Hash del bloque anterior en la cadena', False),
        'timestamp': ('Timestamp de creación (Unix epoch)', True),
        'miner': ('Dirección del minero/validador que creó el bloque', False),
        'difficulty': ('Dificultad de minado (PoW): cuántos intentos se necesitan en promedio para encontrar un hash válido', True),
        'gasLimit': ('Límite máximo de gas permitido en este bloque', True),
        'gasUsed': ('Gas total utilizado por todas las transacciones del bloque', True),
        'baseFeePerGas': ('Fee base por unidad de gas según EIP-1559 (quemada automáticamente)', True),
        'size': ('Tamaño del bloque en bytes', True),
        'nonce': ('Nonce del proof-of-work: número aleatorio usado para minar el bloque', False),
        'mixHash': ('Hash mixto del proof-of-work usado en el algoritmo de consenso', False),
        'stateRoot': ('Hash raíz del árbol Merkle del estado global después de ejecutar este bloque', False),
        'transactionsRoot': ('Hash raíz del árbol Merkle de todas las transacciones en este bloque', False),
        'receiptsRoot': ('Hash raíz del árbol Merkle de los recibos de todas las transacciones', False),
        'logsBloom': ('Bloom filter: estructura de datos probabilística para búsqueda rápida de logs/eventos (512 bytes)', False),
        'extraData': ('Datos extra arbitrarios incluidos por el minero (máx 32 bytes)', False),
        'sha3Uncles': ('Hash Keccak-256 (SHA-3) de la lista de bloques uncle (ommer) incluidos', False),
        'uncles': ('Lista de bloques uncle/ommer: bloques válidos minados casi simultáneamente pero no incluidos en la cadena principal', False),
        'transactions': ('Lista de transacciones incluidas en este bloque', False)
    }
    
    # Iterar sobre los campos del bloque en el orden que vienen
    for field, value in block.items():
        # Obtener definición si existe, sino valores por defecto
        if field in field_definitions:
            description, convert_hex = field_definitions[field]
        else:
            description = field  # Sin descripción, mostrar el nombre del campo
            convert_hex = False  # No convertir por defecto
        
        # Formatear el valor
        if isinstance(value, list):
            display_value = f"{len(value)} elemento(s)"
        elif field == 'timestamp' and isinstance(value, str) and value.startswith('0x'):
            # Convertir timestamp a fecha legible
            timestamp_decimal = hex_to_dec(value)
            timestamp_date = datetime.fromtimestamp(timestamp_decimal).strftime('%Y-%m-%d %H:%M:%S UTC')
            display_value = f"{value} ({timestamp_decimal:,}) → {timestamp_date}"
        elif convert_hex and isinstance(value, str) and value.startswith('0x'):
            decimal = hex_to_dec(value)
            display_value = f"{value} ({decimal:,})"
        else:
            display_value = str(value)[:80] + ('...' if len(str(value)) > 80 else '')
        
        print(f"\n🔹 {field}")
        print(f"   {description}")
        print(f"   Valor: {display_value}")
    
    print("\n" + "=" * 80)
else:
    print("⚠️  No hay datos del bloque para mostrar")

📦 INFORMACIÓN DEL BLOQUE

🔹 hash
   Hash único que identifica este bloque
   Valor: 0x9f5303c4e06e573978156f63b334ef4923d8fe851e4574faa843c1097914eed6

🔹 parentHash
   Hash del bloque anterior en la cadena
   Valor: 0xe6455a3355b273101dedb72f2b05e30806022e680e51716aa97481387383993c

🔹 sha3Uncles
   Hash Keccak-256 (SHA-3) de la lista de bloques uncle (ommer) incluidos
   Valor: 0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a7413f0a142fd40d49347

🔹 miner
   Dirección del minero/validador que creó el bloque
   Valor: 0x2f14582947e292a2ecd20c430b46f2d27cfe213c

🔹 stateRoot
   Hash raíz del árbol Merkle del estado global después de ejecutar este bloque
   Valor: 0xdc7adac1b43457ce43dc57ff85f4c6ca6b3791fb463526acd29908eb48f5a7f2

🔹 transactionsRoot
   Hash raíz del árbol Merkle de todas las transacciones en este bloque
   Valor: 0x56e81f171bcc55a6ff8345e692c0f86e5b48e01b996cadc001622fb5e363b421

🔹 receiptsRoot
   Hash raíz del árbol Merkle de los recibos de todas las transacciones
   Valor: 